# 4-dars: birinchi modellar — kompyuter taxmin qilishni o’rganadi

**Bugun siz:**
- kompyuterga misollar ko’rsatib, uni son taxmin qilishga o’rgatasiz (o’qishga sarflangan soat → imtihon bali);
- Titanic yo’lovchisi omon qolgan-qolmaganini taxmin qiladigan 3 xil model yasaysiz;
- modelning haqiqiy sifatini to’g’ri o’lchashni o’rganasiz (nega «yodlab olgan» model yomon);
- modellarni solishtirib, eng yaxshisini tanlaysiz.

**Asosiy fikr:** model — tajribali ustoz kabi. U ko’p misol ko’radi (**o’rganadi**), keyin yangi holatda javob beradi (**taxmin qiladi**). Sifatini esa faqat u **ilgari ko’rmagan** misollarda tekshirish mumkin.

**Notebook qanday ishlaydi:** matn kataklari — tushuntirish; kod kataklari ▶ (yoki Shift + Enter) bilan ishga tushadi. Kataklarni yuqoridan pastga, tartib bilan ishga tushiring.

⚠️ Avval: **Copy to Drive** (yoki **File → Save a copy in Drive**).

## 0. Kutubxonalar

Yangi kutubxona — **scikit-learn** (`sklearn`). Unda tayyor modellar bor: ularni o’zimiz yozmaymiz, faqat chaqiramiz. Qolganlari 3-darsdagi: `pandas` (jadval), `seaborn` va `matplotlib` (grafik).

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, confusion_matrix, precision_score, recall_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

## 1. Model nima? Kichik misol

Har model ikki ish qiladi:
1. **o’rganadi** — misollarga qaraydi. Kodda: `model.fit(...)` (fit — «moslashtir»);
2. **taxmin qiladi** — yangi holat uchun javob beradi. Kodda: `model.predict(...)`.

Boshlaymiz eng sodda misoldan: 6 o’quvchi imtihondan oldin necha soat o’qigan va necha ball olgan.

In [ ]:
oqish = pd.DataFrame({
    "soat": [1, 2, 3, 4, 6, 7],
    "ball": [35, 45, 50, 65, 80, 85],
})
print(oqish)

sns.scatterplot(data=oqish, x="soat", y="ball")   # har nuqta — bitta o'quvchi
plt.title("Soat va ball")
plt.show()

Nuqtalar deyarli bir chiziqda: ko’proq o’qigan — ko’proq ball. **Chiziqli regressiya** modeli aynan shunday chiziqni topadi: nuqtalar orasidan eng to’g’ri o’tadigan chiziq.

Kodda:
- `soat` ustuni — **nimaga qarab taxmin qilamiz** (3-darsdagi X). Ikki qavs: `oqish[["soat"]]`;
- `ball` ustuni — **nimani topmoqchimiz** (y). Bitta qavs: `oqish["ball"]`.

In [ ]:
X_s = oqish[["soat"]]       # nimaga qarab (jadval)
y_s = oqish["ball"]         # nimani topamiz (ustun)

model = LinearRegression()  # bo'sh model yasadik
model.fit(X_s, y_s)         # o'rgandi: 6 ta misolga qarab chiziqni topdi

print(f"ball = {model.intercept_:.1f} + {model.coef_[0]:.1f} × soat")

Chiziqni chizib ko’ramiz: ko’k nuqtalar — o’quvchilar, qizil chiziq — model topgan formula.

In [ ]:
sns.scatterplot(data=oqish, x="soat", y="ball")                  # o'quvchilar
plt.plot(oqish["soat"], model.predict(X_s), color="red")         # model chizig'i
plt.title("Model topgan chiziq")
plt.show()

Model formula topdi: **ball = 27.1 + 8.6 × soat**. Ya’ni har qo’shimcha soat o’rtacha 8.6 ball beradi, o’qimasa ham ~27 ball. Endi taxmin qildiramiz: 5 soat o’qigan o’quvchi necha ball oladi?

`predict` ga ham jadval berish kerak (`X_s` kabi, ustun nomi `soat`).

In [ ]:
yangi = pd.DataFrame({"soat": [5]})    # yangi o'quvchi: 5 soat o'qigan
print(model.predict(yangi))            # taxmin: [70.]

**Sinab ko’ring.** 10 soat o’qigan o’quvchi necha ball oladi? Avval taxmin qiling: 100 dan oshib ketadimi?

In [ ]:
# Avval taxmin qiling, keyin shu yerga yozib tekshiring

Model 112.9 ball dedi, ammo ball 100 dan oshmaydi! Model faqat chiziqni davom ettiradi, ball cheklangan ekanini **bilmaydi**. Model hamma narsani bilmaydi, u faqat ko’rgan misollaridan o’rganadi. Shuning uchun uning javobiga o’ylab qarash kerak.

**Mustaqil ish 1.** 8 soat o’qigan o’quvchi uchun taxmin qiling. Namuna — yuqoridagi 5 soatlik katak.

In [ ]:
# TODO: 8 soat o'qigan o'quvchi uchun taxmin
yangi8 = pd.DataFrame({"soat": [8]})
taxmin8 = ...

assert taxmin8 is not ..., "taxmin8 = ... o'rniga model.predict(...) yozing"
print(round(float(taxmin8[0]), 1))   # 95.7 chiqishi kerak

### Model qani?

«Model» deymiz, lekin uni ko’ra olamizmi? Ko’ra olamiz. Kodda `model` — oddiy o’zgaruvchi. Ichiga qaraymiz.

- **O’rganishdan oldin** model bo’sh: unda hech qanday topilgan son yo’q.
- **O’rganishdan keyin** (`fit`) uning ichida **topilgan sonlar** paydo bo’ladi: bu holatda ikkita — `intercept_` (27.1) va `coef_` (8.6). Nomi oxiridagi pastki chiziq (`coef_`) — «o’rganishdan keyin paydo bo’ladi» degan belgi.

In [ ]:
bosh_model = LinearRegression()                                         # hali o'rganmagan model
print("o'rganishdan oldin sonlar bormi?", hasattr(bosh_model, "coef_"))   # False — hali bo'sh
print("o'rganishdan keyin sonlar bormi? ", hasattr(model, "coef_"))       # True

print("model ichidagi sonlar:", model.intercept_.round(1), model.coef_.round(1))   # 27.1 va [8.6]

**Model — shu ikki son va ularni ishlatadigan qoida.** `fit` shu sonlarni topadi. `predict` ularni formulaga qo’yadi: 27.1 + 8.6 × soat. Boshqa misollar ko’rsatsak — boshqa sonlar chiqadi, ya’ni **boshqa model** bo’ladi.

**Sinab ko’ring.** Faqat birinchi 3 o’quvchini ko’rsatib yangi model yasang (`X_s[:3]` va `y_s[:3]`): ichidagi sonlar (`coef_`) o’zgaradimi?

In [ ]:
# Avval taxmin qiling, keyin shu yerga yozib tekshiring

**Model — fayl ham bo’la oladi.** O’rganilgan sonlarni faylga yozib qo’yish va keyin boshqa joyda qayta ochish mumkin. Kursning 1-loyihasida («Ko’z») aynan shunday qilamiz: modelni o’qitamiz, faylga yozamiz va saytga olib o’tamiz.

In [ ]:
import joblib

joblib.dump(model, "model.joblib")                 # modelni faylga yozdik
ochilgan = joblib.load("model.joblib")             # fayldan qayta ochdik
print(ochilgan.predict(pd.DataFrame({"soat": [5]})))   # o'sha taxmin: [70.]

**Model qanchalik adashadi?** Har o’quvchi uchun «taxmin − haqiqiy ball» farqini olamiz, keyin o’rtachasini hisoblaymiz. Bu **MAE** (o’rtacha xato): kichik bo’lsa — yaxshi.

In [ ]:
taxminlar = model.predict(X_s)                    # 6 ta o'quvchi uchun taxmin
print(round(mean_absolute_error(y_s, taxminlar), 1))   # taxmin va haqiqiy ball farqining o'rtachasi

Model o’rtacha 1.9 ball adashadi. **Lekin diqqat:** biz modelni o’zi o’rgangan 6 o’quvchida tekshirdik. Bu xuddi imtihonda darslikdagi savollarning o’zini berishga o’xshaydi: ball yuqori bo’ladi, ammo bilimni ko’rsatmaydi. Pastda buni to’g’rilaymiz.

**Ikki xil taxmin:**

| | Nimani taxmin qiladi | Misol |
|---|---|---|
| **Regressiya** | son | imtihon bali, narx |
| **Klassifikatsiya** | toifa (guruh) | omon qoldimi (ha/yo’q), rasmdagi raqam |

## 2. Titanic: klassifikatsiya

Endi toifa taxmin qilamiz: **yo’lovchi omon qolganmi?** Jadvalni 3-darsdagidek tayyorlaymiz:
- `age` dagi bo’sh joylarni sinf mediani bilan to’ldiramiz;
- `sex` — matn (`male`/`female`), model esa faqat son tushunadi. Shuning uchun yangi `ayol` ustuni: ayol bo’lsa 1, aks holda 0. `==` — 3-darsdagi filtrdagi kabi «tengmi?» deb so’raydi.

In [ ]:
df = sns.load_dataset("titanic")
df["age"] = df["age"].fillna(df.groupby("pclass")["age"].transform("median"))   # 3-dars
df["ayol"] = (df["sex"] == "female").astype(int)                                # ayol -> 1, erkak -> 0

X = df[["pclass", "ayol", "age", "fare"]]   # nimaga qarab taxmin qilamiz
y = df["survived"]                          # nimani topamiz: omon qolganmi
print(X.shape, y.shape)
X.head()

### Train va test: ikkiga bo’lamiz

Modelni qanday tekshiramiz? Imtihonni eslang. O’qituvchi darsda yechilgan savollarning o’zini bersa, hamma yaxshi baho oladi, ammo bu bilimni ko’rsatmaydi. Shuning uchun ma’lumotni ikkiga bo’lamiz:
- **train** (o’quv, 80%) — model shu misollardan o’rganadi;
- **test** (imtihon, 20%) — model uni **hech qachon ko’rmagan**. Sifat shu yerda o’lchanadi.

`random_state=42` — aralashtirish hammada bir xil chiqishi uchun (raqam o’zgarmaydi).

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
# Xtr, ytr — o'quv jadvali va javoblari; Xte, yte — imtihon jadvali va javoblari
print("o'quv:", len(Xtr), "| imtihon:", len(Xte))

### Eng oddiy «model»: hamma halok bo’ladi

Murakkab modelning yaxshi-yomonligini tushunish uchun avval eng oddiy qoidaga qaraymiz: har bir yo’lovchi uchun «halok bo’ladi» (0) deb javob beramiz. U qanchalik to’g’ri chiqadi?

**Aniqlik (accuracy)** — to’g’ri javoblar ulushi. Test javoblari (`yte`) orasida nol (halok) ulushi — shu qoidaning aniqligi.

**Mustaqil ish 2.** `(yte == 0)` — har yo’lovchi uchun True/False (halokmi?). Uning `.mean()` si — True ulushi.

In [ ]:
# TODO: "hamma halok bo'ladi" qoidasining aniqligi: yte nolga teng bo'lganlar ulushi
oddiy_aniqlik = ...

assert oddiy_aniqlik is not ..., "oddiy_aniqlik = ... o'rniga hisoblang"
print(f"Hamma halok: {100 * oddiy_aniqlik:.1f}%")   # 58.7% chiqishi kerak

Hech narsa o’rganmagan qoida ham **58.7%** to’g’ri javob beradi. Demak, har bir haqiqiy model bu sondan yuqori chiqishi kerak, aks holda u foydasiz.

## 3. Uchta model

Hamma modellar bir xil ishlaydi: **`fit(Xtr, ytr)`** — o’quv misollaridan o’rgan; **`score(Xte, yte)`** — test misollarida aniqlikni ber. Faqat modelning nomi o’zgaradi.

**1-model: logistik regressiya.** Har ustunga «ball» beradi (masalan, ayol bo’lish — plyus, 3-sinf — minus), hammasini qo’shib, «omon qoladi» yoki «qolmaydi» deb qaror qiladi. Nomida «regressiya» bor, lekin u toifa taxmin qiladi.

In [ ]:
logistik = LogisticRegression(max_iter=1000)   # max_iter — nechta urinish; ko'proq berdik
logistik.fit(Xtr, ytr)                         # o'quv misollaridan o'rgandi

print("o'quv aniqligi: ", round(logistik.score(Xtr, ytr), 3))
print("imtihon aniqligi:", round(logistik.score(Xte, yte), 3))   # eng muhim son

Imtihon aniqligi ~80%: «hamma halok» qoidasidan (58.7%) ancha yaxshi.

**Bu modelning ichida nima bor?** Ham xuddi shunday sonlar: har ustunga bittadan «ball».

In [ ]:
print(pd.Series(logistik.coef_[0], index=X.columns).round(2))   # har ustunning "bali"

O’qish: `ayol` ning bali **+2.46** — ayol bo’lish omon qolishga eng katta plyus; `pclass` ning bali **−1.05** — sinf raqami kattalashgani sari (arzonroq bilet) omon qolish kamayadi. `age` va `fare` ballari 0 ga yaqin: ular har bir birlik (1 yosh, 1 dollar) uchun hisoblangan. Model qaror qilganda shu ballarni qo’shadi.

**Boshqa modellarda «ichi» boshqacha:**

| Model | Ichida nima bor |
|---|---|
| chiziqli va logistik regressiya | topilgan sonlar (ballar) |
| qaror daraxti | «ha/yo’q» savollar zanjiri |
| KNN | o’quv misollarining o’zi (xotirada saqlanadi) |

Endi bitta yo’lovchi uchun taxmin: `predict` ga bitta qator beramiz.

**Sinab ko’ring.** Ikkita yo’lovchi: (1) ayol, 1-sinf, 30 yosh, bilet 100; (2) erkak, 3-sinf, 25 yosh, bilet 8. Ustunlar tartibi `X` dagidek: `pclass, ayol, age, fare`. Model qanday taxmin qiladi? 1 — omon qoladi, 0 — yo’q.

In [ ]:
# Avval taxmin qiling, keyin shu yerga yozib tekshiring

**2-model: qaror daraxti.** «Ha/yo’q» savollar zanjiri: «ayolmi? — ha → 1-2-sinfmi? — ha → …». Pastda uning birinchi savollarini ko’rasiz.

In [ ]:
kichik = DecisionTreeClassifier(max_depth=2, random_state=42).fit(Xtr, ytr)   # faqat 2 qavat savol
print(export_text(kichik, feature_names=list(X.columns)))

Birinchi savol — «ayolmi?» (`ayol <= 0.50` — ayol emas, ya’ni erkak). Daraxt eng foydali savoldan boshlaydi.

**Mustaqil ish 3.** Cheklovsiz daraxt yasang: `DecisionTreeClassifier(random_state=42)`. Uni `fit(Xtr, ytr)` bilan o’qiting va **ikkala** aniqlikni (`o’quv`, `imtihon`) chiqaring. Namuna — logistik regressiya katagi.

In [ ]:
daraxt = DecisionTreeClassifier(random_state=42)
# TODO: daraxtni o'qiting (fit), keyin o'quv va imtihon aniqligini chiqaring (score)
...

assert hasattr(daraxt, "tree_"), "daraxt.fit(Xtr, ytr) ni yozing"

O’quv aniqligi deyarli 100%, imtihonda esa ancha past. Nega bunday? Pastda ko’ramiz. Hozircha uchinchi model.

**3-model: KNN.** Nomini ochamiz:
- **NN** = *Nearest Neighbor* — «eng yaqin qo’shni». 1-darsdagi **1-NN** ni eslang: yangi rasmga eng o’xshash **bitta** rasmni topib, uning javobini olardik;
- **KNN** = *K Nearest Neighbors* — «**k ta** eng yaqin qo’shni». Bitta emas, k ta eng o’xshash misolga qaraydi va **ko’pchilik ovozi** bilan javob beradi. Masalan, k = 5: eng o’xshash 5 yo’lovchidan 3 tasi omon qolgan bo’lsa, javob — «omon qoladi».

«Yaqin» — 1–2-darslardagi **masofa** bilan o’lchanadi. Kodda: `n_neighbors=5` — 5 ta eng yaqin yo’lovchi.

**Mustaqil ish 4.** `KNeighborsClassifier(n_neighbors=5)` yasang, o’qiting va ikkala aniqlikni chiqaring.

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)
# TODO: knn ni o'qiting (fit), keyin o'quv va imtihon aniqligini chiqaring
...

assert hasattr(knn, "n_samples_fit_"), "knn.fit(Xtr, ytr) ni yozing"

Hozirgi holat (imtihon aniqligi):

| Model | Imtihon |
|---|---|
| hamma halok (modelsiz) | 58.7% |
| logistik regressiya | ~80% |
| qaror daraxti | ~76% |
| KNN | ~71% |

KNN eng yomon chiqdi, daraxtning o’quv va imtihon sonlari esa juda farqli. Ikkalasining sababini topamiz.

## 4. Yodlab olish: overfitting

Cheklovsiz daraxt o’quv misollarini **yodlab oldi**: har yo’lovchi uchun o’z «qoidasi» bor. Shuning uchun o’quvda 98%, lekin yangi yo’lovchida adashadi (76%). Bu **overfitting** («ortiqcha moslashish»): darsdagi savollarni yodlab olib, yangi savolda adashadigan o’quvchiga o’xshaydi.

Daraxtning **chuqurligi** (`max_depth`) — nechta savolni ketma-ket berishi. Kichik chuqurlik — oddiy model; katta — murakkab, yodlab oladigan model. Har chuqurlik uchun o’quv va imtihon aniqligini chizamiz.

In [ ]:
chuqurliklar = [1, 2, 3, 4, 5, 6, 8, 10, 15, 20]
oquv, imtihon = [], []
for d in chuqurliklar:
    t = DecisionTreeClassifier(max_depth=d, random_state=42).fit(Xtr, ytr)
    oquv.append(t.score(Xtr, ytr))        # o'quvdagi aniqlik
    imtihon.append(t.score(Xte, yte))     # imtihondagi aniqlik

plt.plot(chuqurliklar, oquv, marker="o", label="o'quv")
plt.plot(chuqurliklar, imtihon, marker="o", label="imtihon")
plt.xlabel("Daraxt chuqurligi")
plt.ylabel("Aniqlik")
plt.title("Chuqurlik va aniqlik")
plt.legend()
plt.show()

**O’qish:**
- **o’quv** chizig’i hamma vaqt o’sadi (0.79 dan 0.98 gacha): model misollarni tobora yaxshi yodlaydi;
- **imtihon** chizig’i 3-chuqurlikdan keyin deyarli o’smaydi (~0.80), 15 va undan chuqurda esa 0.76 ga tushadi;
- ikki chiziq orasidagi **bo’shliq** kattalashadi: murakkab modelning o’quvdagi ustunligi yolg’on. Bu — overfitting;
- chuqurlik 1 da ikkalasi past: model juda oddiy, yetarli o’rgana olmadi (**underfitting**).

Xulosa: chuqurlik 3 yetarli. Bundan murakkabroq daraxt imtihonda foyda bermaydi, faqat yodlab oladi.

**Mustaqil ish 5.** Chuqurligi 3 bilan cheklangan daraxt yasang (`max_depth=3`), o’qiting va **imtihon** aniqligini chiqaring.

In [ ]:
# TODO: max_depth=3 bilan daraxt yasab o'qiting va imtihon aniqligini chiqaring
oddiy_daraxt = DecisionTreeClassifier(max_depth=3, random_state=42)
...

assert hasattr(oddiy_daraxt, "tree_"), "oddiy_daraxt.fit(Xtr, ytr) ni yozing"

Oddiyroq daraxt imtihonda cheklovsizdan **yaxshiroq** chiqdi (~0.80 va 0.76). Model qancha murakkab bo’lsa, shuncha yaxshi emas. **O’quv aniqligiga ishonmang, imtihon aniqligiga qarang.**

## 5. Aniqlik hamma gapni aytmaydi

Model «omon qoladi» desa, bu to’g’rimi? Va qanday xato qiladi? Buni ko’rish uchun **chalkashlik jadvali** (confusion matrix): 2 × 2 jadval — taxmin va haqiqat.

In [ ]:
taxmin = logistik.predict(Xte)           # test yo'lovchilari uchun taxmin
print(confusion_matrix(yte, taxmin))     # qatorlar — haqiqat, ustunlar — taxmin

Natija:

| | Model «halok» dedi | Model «omon qoladi» dedi |
|---|---|---|
| **Haqiqatda halok** | **91** (to’g’ri) | **14** (xato: halok, lekin «omon» dedi) |
| **Haqiqatda omon qolgan** | **21** (xato: omon, lekin «halok» dedi) | **53** (to’g’ri) |

To’g’ri javoblar diagonalda: 91 + 53 = 144 ta, 179 tadan = 80.4%. Ikki turdagi xatoni alohida o’lchaymiz.

Misol: yo’qolgan mushuklarni qidiryapsiz. Siz 67 ta hayvonni «mushuk» deb ko’rsatdingiz, ularning 53 tasi haqiqatan mushuk edi.
- **Precision (aniqlik-tozaligi):** «mushuk» degan javoblaringizning nechasi to’g’ri? 53 / 67.
- **Recall (to’liqlik):** hamma mushuklarning nechasini topdingiz? Jami mushuk 74 ta: 53 / 74.

Bizda «mushuk» — «omon qoladi».

In [ ]:
print("precision:", round(precision_score(yte, taxmin), 3))   # "omon qoladi" degan javoblarning to'g'risi
print("recall:   ", round(recall_score(yte, taxmin), 3))      # haqiqatan omon qolganlarning topilgani

**Sinab ko’ring.** Yuqoridagi jadvaldan precision va recall ni qo’lda hisoblang: `53 / (53 + 14)` va `53 / (53 + 21)`. Kod bilan chiqqan sonlar bilan bir xil bo’ladimi?

In [ ]:
# Avval taxmin qiling, keyin shu yerga yozib tekshiring

**Qaysi biri muhimroq?** Vaziyatga bog’liq:
- kasallikni aniqlashda **recall** muhim: bitta kasalni o’tkazib yubormaslik kerak;
- spam filtrida **precision** muhim: yaxshi xatni spam deb yo’qotib qo’ymaslik kerak.

Shuning uchun faqat aniqlikka qarab model tanlamang.

## 6. Bitta bo’linish tasodif bo’lishi mumkin: cross-validation

Biz ma’lumotni bir marta bo’ldik va imtihon aniqligini oldik. Ammo tasodifan «oson» yo’lovchilar testga tushgan bo’lishi mumkin. Ishonchliroq usul — **cross-validation**: ma’lumotni 5 bo’lakka bo’lamiz va 5 marta o’qitamiz; har safar boshqa bo’lak imtihon bo’ladi. Keyin 5 ta natijaning o’rtachasini olamiz.

In [ ]:
ballar = cross_val_score(LogisticRegression(max_iter=1000), X, y, cv=5)   # cv=5 — 5 bo'lak
print(ballar.round(3))                  # 5 marta imtihon natijasi
print("o'rtacha:", ballar.mean().round(3))

Besh natija bir-biridan biroz farq qiladi: bitta bo’linishga ishonib bo’lmasligining sababi shu. O’rtacha (~0.79) ishonchliroq son.

## 7. KNN nega yomon chiqdi?

KNN «eng yaqin» yo’lovchilarni **masofa** bo’yicha topadi (1–2-darslardagi masofani eslang: farqlarni qo’shadi). Ustunlarimiz esa har xil o’lchamda:
- `ayol`: 0 yoki 1;
- `pclass`: 1 dan 3 gacha;
- `age`: 0 dan 80 gacha;
- `fare`: 0 dan 500 gacha.

Narx (`fare`) masofani «bosib ketadi»: jins va sinf deyarli sezilmaydi. Xuddi bo’yni santimetrda, vaznni grammda o’lchab, ikki odamni solishtirgandek: vazn hamma narsani hal qiladi.

Yechim: `StandardScaler` — har ustunni bir xil o’lchamga keltiradi, keyin KNN. `make_pipeline` ikkalasini ketma-ket bog’laydi.

In [ ]:
knn_toza = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))
knn_toza.fit(Xtr, ytr)    # avval o'lchamni tenglaydi, keyin KNN o'rganadi
print("imtihon aniqligi:", round(knn_toza.score(Xte, yte), 3))

KNN ~71% dan ~81% ga ko’tarildi: modeldagi muammo uning o’zida emas, ma’lumotni tayyorlashda edi.

### Hammasini solishtiramiz

Modellar ro’yxatini yasab, har biri uchun imtihon aniqligi va cross-validation o’rtachasini hisoblaymiz. Natijalarni **pandas jadvaliga** yig’amiz (3-dars).

In [ ]:
modellar = {
    "Logistik regressiya": LogisticRegression(max_iter=1000),
    "Daraxt (chuqurlik 3)": DecisionTreeClassifier(max_depth=3, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "KNN + masshtab": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
}

satrlar = []                                   # har model uchun bitta qator
for nom, m in modellar.items():
    m.fit(Xtr, ytr)
    test = m.score(Xte, yte)                   # imtihon aniqligi
    cv = cross_val_score(m, X, y, cv=5).mean() # 5 marta imtihon o'rtachasi
    satrlar.append([nom, test, cv])
print(len(satrlar), "ta model baholandi")

**Mustaqil ish 6.** `satrlar` ro’yxatidan jadval yasang: `pd.DataFrame(satrlar, columns=["model", "test", "cv"])`. Eng yaxshi model tepada turishi uchun `test` bo’yicha tartiblang: `.sort_values("test", ascending=False)`.

In [ ]:
# TODO: satrlar dan jadval yasang va test bo'yicha kamayish tartibida saralang
natijalar = ...

assert natijalar is not ..., "natijalar = ... o'rniga jadvalni yozing"
natijalar.round(3)

**O’qish:** tepada — imtihon aniqligi eng yuqori model. `test` va `cv` bir-biriga yaqin bo’lsa, natijaga ishonish mumkin. Modelni faqat bitta songa qarab tanlamang: imtihon, cross-validation, o’quv va imtihon farqi, xatolar turi (confusion matrix) hammasi muhim.

## Uy ishi: eng yaxshi modelni tanlang va asoslang

Yuqoridagi `natijalar` jadvaliga qarab:
1. Qaysi model eng yaxshi? Tanlang.
2. 3–4 gapda **raqamlar bilan** asoslang: imtihon va cross-validation aniqligi, «hamma halok» qoidasi (58.7%) bilan solishtirish.
3. Cheklovsiz daraxt nega tanlanmadi? (o’quv va imtihon aniqligidagi farqni yozing).
4. Tanlangan model uchun `confusion_matrix`, `precision` va `recall` ni hisoblab, qaysi xato qimmatroq deb o’ylaysiz? Nega?

Yozma javobingizni notebookning oxirgi katagida (matn katagi) yozing.

**Javobingiz:** shu katakni ikki marta bosib yozing.

## Chuqurroq (ixtiyoriy)

3-darsda o’z datasetingiz bo’yicha EDA qildingiz. Shu jadvaldan model yasang: seaborn’ning `penguins` datasetida pingvin turini (`species`) taxmin qiling.

In [ ]:
ping = sns.load_dataset("penguins").dropna()                     # bo'sh joyli qatorlarni olib tashladik
Xp = ping[["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]]   # tumshuq, qanot, vazn
yp = ping["species"]                                             # nimani topamiz: turi

Xp_tr, Xp_te, yp_tr, yp_te = train_test_split(Xp, yp, test_size=0.2, random_state=42)
model_p = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)).fit(Xp_tr, yp_tr)
print("imtihon aniqligi:", round(model_p.score(Xp_te, yp_te), 3))

**Vazifa:** `KNeighborsClassifier` o’rniga `DecisionTreeClassifier(max_depth=3)` yoki `LogisticRegression(max_iter=1000)` qo’ying (masshtab shart emas) va natijani solishtiring. Qaysi biri yaxshi?

## Yakun

Bugun siz:
- modelning ikki ishini bildingiz: `fit` (o’rganadi) va `predict` (taxmin qiladi);
- son (regressiya) va toifa (klassifikatsiya) taxmin qildingiz;
- ma’lumotni train va testga bo’lib, sifatni imtihonda o’lchadingiz;
- yodlab olishni (overfitting) ko’rdingiz;
- accuracy, precision, recall va cross-validation nima ekanini tushundingiz;
- modellarni jadvalda solishtirdingiz.

Keyingi darsda eng yaxshi usullardan biriga chuqurroq kiramiz: modelning o’zi qanday o’rganishini o’zimiz yozamiz (loss va gradient descent).